## 2. The Architecture: Attention and Wavelets
Standard Physics-Informed Neural Networks (PINNs) suffer from **Spectral Bias**—they naturally prefer learning low-frequency, smooth functions. The HH equations, however, are dominated by high-frequency, violent voltage spikes. If a standard MLP tries to learn this, it will either smooth out the spike or oscillate uncontrollably. 

We solve this using a two-pronged architectural approach:

### A. Global Temporal Awareness (Self-Attention)
The Transformer's Self-Attention mechanism computes how every single time point relates to every other time point in the sequence:
$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$
This allows the network to instantly correlate the slow buildup of the gating variables at $t=10$ms with the explosive voltage spike at $t=35$ms, entirely bypassing the need for an ODE solver.

### B. The Anti-Stiffness Weapon (Mexican Hat Wavelet)
To defeat Spectral Bias, we replace standard global activations (like $\tanh$) inside the Feed-Forward block with the **Mexican Hat Wavelet**:
$$ \psi(x) = (1 - x^2) \exp\left(-\frac{x^2}{2}\right) $$
Unlike $\tanh$, which affects the output everywhere, a wavelet is localized. It stays near zero during the flat resting potential, and sharply "fires" only when the HH dynamics demand an action potential. The network learns to stretch and shift these wavelets to perfectly capture high-frequency stiff dynamics without corrupting the smooth areas.

## 3. The Seq2Seq Physics Loss (Finite Differences)
Because we have eliminated the ODE solver, we can no longer evaluate the physics at a single point in time and step forward. We must enforce the Hodgkin-Huxley physics directly across the entire predicted sequence at once.

### The Physics Residual
We need to know the predicted temporal derivative ($du/dt$) of our sequence. Since doing continuous auto-differentiation through complex Attention layers is computationally brutal, we use **Central Finite Differences** across our discrete sequence $U_{pred}$:
$$ \frac{d u_i}{dt} \approx \frac{u_{i+1} - u_{i-1}}{2\Delta t} $$

We pass our interior sequence points through the exact HH physical equations—let's call that mathematical operator $\mathcal{F}(u)$—and penalize any difference between our sequence's actual derivative and the theoretical derivative:
$$ \mathcal{L}_{phys} = \frac{1}{N} \sum_{i=2}^{N-1} \left\| \frac{u_{i+1} - u_{i-1}}{2\Delta t} - \mathcal{F}(u_i) \right\|^2 $$

*Note: Just like in the PI-NODE-SR framework, we still divide this residual by our Scale Factors ($s_j$) to ensure the massive voltage gradients do not drown out the tiny gating gradients!*

In [1]:
using  Flux,SciMLSensitivity, Optimization, OptimizationOptimisers, Statistics, Random, ComponentArrays, Zygote

In [2]:

using CSV, DataFrames

file_path = raw"C:\nirbhay\Downloads\Neural_Pinnformmer\Synthetic_Data\single_spike_noisy_data.csv"


HH_data = CSV.read(file_path, DataFrame)



# 1. Correct the DataFrame mapping
df_ordered = DataFrame(
    timestamp = HH_data.timestamp,
    V = HH_data.V,
    n = HH_data.n, 
    m = HH_data.m,
    h = HH_data.h  
)

# 2. Extract into Float32 arrays for Lux.jl / Zygote.jl
t_train = Float32.(df_ordered.timestamp)

# 3. Extract states and transpose to get the required shape: (4, Total_Time_Steps)
z_synthetic = Float32.(Matrix(df_ordered[:, [:V, :n, :m, :h]]))'

println("Time array shape: ", size(t_train))
println("State matrix shape: ", size(z_synthetic))

Time array shape: (1469,)
State matrix shape: (4, 1469)


In [31]:
z_synthetic

4×1469 adjoint(::Matrix{Float32}) with eltype Float32:
 -65.0   -64.8363     -64.7979    …  -57.3823    -57.4143    -57.4497
   0.6     0.599784     0.599864       0.453623    0.453502    0.453245
   0.05    0.0502886    0.050523       0.112056    0.112553    0.112654
   0.32    0.320059     0.32014        0.395085    0.395061    0.39517

In [32]:
using Statistics

In [33]:
total_steps = 1469
dt = 0.01f0
z_synthetic = rand(Float32, 4, total_steps)

4×1469 Matrix{Float32}:
 0.493873   0.391318  0.651754  0.843443  …  0.424742  0.686538  0.101988
 0.0848667  0.393449  0.437219  0.117639     0.389823  0.308528  0.761494
 0.857736   0.294206  0.790117  0.128094     0.18191   0.169446  0.54225
 0.1063     0.229838  0.531114  0.531181     0.276152  0.957269  0.439469

In [34]:
# Define your sliding window length
window_lens = 50
# sliding window
t_reletive = reshape(Float32.(0:dt:(window_len-1)*dt), (1,window_len,1))

1×50×1 Array{Float32, 3}:
[:, :, 1] =
 0.0  0.01  0.02  0.03  0.04  0.05  …  0.44  0.45  0.46  0.47  0.48  0.49

In [35]:
print("Initial Dimensions")
println("z_synthetic = " , size(z_synthetic))
print("t_reletive = " , size(t_reletive))

Initial Dimensionsz_synthetic = (4, 1469)
t_reletive = (1, 50, 1)

In [36]:
# Silding windows transfromation 
# random valid k 
max_k = total_steps-window_lens+1
k = rand(1:max_k)
# : left is rows and right in columns
# all row and columns ( k to k+4 )
z_traget =z_synthetic[: , k:(k+window_lens-1)]


4×50 Matrix{Float32}:
 0.0920238  0.743593  0.282918  0.739463  …  0.933255  0.517697   0.393516
 0.109635   0.089651  0.470775  0.110343     0.196763  0.959796   0.0839075
 0.934553   0.423465  0.875193  0.771856     0.124935  0.0746062  0.754445
 0.0794151  0.710271  0.806117  0.457515     0.618004  0.450193   0.563602

In [37]:
z_window = reshape(z_traget, (4, window_len, 1))

4×50×1 Array{Float32, 3}:
[:, :, 1] =
 0.0920238  0.743593  0.282918  0.739463  …  0.933255  0.517697   0.393516
 0.109635   0.089651  0.470775  0.110343     0.196763  0.959796   0.0839075
 0.934553   0.423465  0.875193  0.771856     0.124935  0.0746062  0.754445
 0.0794151  0.710271  0.806117  0.457515     0.618004  0.450193   0.563602

In [38]:
println("--- WINDOW SLICE AT INDEX k = $k ---")
println("Sliced target window shape : ", size(z_window)) # (4, 50, 1)
println("")

--- WINDOW SLICE AT INDEX k = 420 ---
Sliced target window shape : (4, 50, 1)



In [39]:
# Simulating a mock neural network output block of shape (4, 50, 1)
u_hat = rand(Float32, 4, window_len, 1)

4×50×1 Array{Float32, 3}:
[:, :, 1] =
 0.607532  0.978082    0.845886   0.178209   …  0.526509  0.0525407  0.671106
 0.911799  0.983328    0.418019   0.254018      0.62557   0.701409   0.128008
 0.456145  0.05942     0.0951677  0.819742      0.165763  0.0220222  0.658016
 0.136911  0.00466442  0.925099   0.0129973     0.181502  0.788922   0.401749

In [40]:
# Loss Component 1: Data Loss over the sliced window
loss_data = mean((u_hat .- z_window).^2)

0.18149547f0

In [41]:
# Loss Component 2: Dynamic Initial Condition Anchor
# We pull the first predicted token and match it back to index 'k' of the raw data
u_init_pred = u_hat[:, 1, 1]
u_init_true = z_synthetic[:, k] # Exactly matches the entry point of the window

loss_ic = mean((u_init_pred .- u_init_true).^2)

println("--- IDENTITY & ANCHOR VERIFICATION ---")
println("Prediction Initial Token Shape       : ", size(u_init_pred)) # (4,)
println("True Data Anchor Shape at index ($k)  : ", size(u_init_true)) # (4,)
println("Data Loss                            : ", loss_data)
println("Initial Condition Loss               : ", loss_ic)

--- IDENTITY & ANCHOR VERIFICATION ---
Prediction Initial Token Shape       : (4,)
True Data Anchor Shape at index (420)  : (4,)
Data Loss                            : 0.18149547
Initial Condition Loss               : 0.285349


# ------------------------------------------------

In [42]:
total_steps = size(z_synthetic,2)
println("total_steps = ", total_steps)
# exact number of windows avilable here
num_windows = div(total_steps-window_len, stride)+1
println("num_windows = ", num_windows)


total_steps = 1469


LoadError: MethodError: no method matching div(::Int64, ::typeof(stride), ::RoundingMode{:ToZero})
The function `div` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  div(::Number, [91m::Missing[39m, ::RoundingMode)
[0m[90m   @[39m [90mBase[39m [90m[4mmissing.jl:130[24m[39m
[0m  div(::Integer, [91m::Rational[39m, ::RoundingMode)
[0m[90m   @[39m [90mBase[39m [90m[4mrational.jl:507[24m[39m
[0m  div(::Real, [91m::Static.StaticInteger{Y}[39m, ::RoundingMode) where Y
[0m[90m   @[39m [36mStatic[39m [90mC:\Users\nirbh\.julia\packages\Static\TjBVO\src\[39m[90m[4mStatic.jl:511[24m[39m
[0m  ...


In [ ]:
<!-- ![Tensor Visualization](images/tensor.png) -->

In [ ]:
# Allocate Static Arrays for Window Tensors and Absolute Tracking Anchors
# shape passport (features = 4 , sequence_length = 50 , batch_size/num_windows)
z_windows = zeros(Float32,4, window_len,num_windows )

4×50×284 Array{Float32, 3}:
[:, :, 1] =
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0

[:, :, 2] =
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0

[:, :, 3] =
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.

In [ ]:
z_windows

LoadError: UndefVarError: `z_windows` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
# Get the total number of elements allocated in memory
length(z_windows)

56800

In [ ]:
# 3. Construct Uniform, Local Relative Time Grid
# Shape passport : (1 , sequence_length = 50, 1)
""" 
Your main matrix and your time matrix have two totally distinct jobs in a Neural Network:z_windows (4, 50, 256) is your DATA:
It holds your 4 physical features ($V, n, m, h$) across 50 timesteps for 256 different window batches.
Every single one of these 256 windows contains completely different numbers because they are cut from different places in your timeline.
t_relative (1, 50, 1) is your constant COORDINATE SYSTEM: It is just a ruler. 
It simply says: "No matter which window you are looking at, the first step is step 0.00, the second step is step 0.01,
and the last step is step 0.49." Every single one of those 256 windows shares this exact same clock.
"""
t_relative = reshape(Float32.(0:dt:(window_len-1)*dt), (1, window_len, 1))

1×50×1 Array{Float32, 3}:
[:, :, 1] =
 0.0  0.01  0.02  0.03  0.04  0.05  …  0.44  0.45  0.46  0.47  0.48  0.49

In [ ]:
window_start_indices = zeros(Int, num_windows)
println("-----------")

-----------


In [ ]:
# 4. Populate the Data Pipeline via Shifting Window Slices
for i in 1:num_windows
    # Calculate the exact bounding boundaries for the current window slice
    start_idx = (i - 1) * stride + 1
    end_idx = start_idx + window_len - 1
    
    # Extract the 4-feature state chunk and store it as the i-th batch element
    z_windows[:, :, i] = z_synthetic[:, start_idx:end_idx]
    
    # Drop the breadcrumb trail tracking the absolute dataset index origin
    window_start_indices[i] = start_idx
end

In [ ]:
# ====================================================================
# PIPELINE DIAGNOSTIC VERIFICATION
# ====================================================================
println("--- SLIDING WINDOW PIPELINE DIAGNOSTICS ---")
println("Total Source Timesteps     : ", total_steps)
println("Generated Windows Count    : ", num_windows)
println("Relative Time Matrix Shape : ", size(t_relative))    # Expected: (1, 50, 1)
println("Tensor Windows Array Shape : ", size(z_windows))     # Expected: (4, 50, num_windows)
println("Saved Tracking Anchors     : ", length(window_start_indices))
println("\nFirst 5 Window Absolute Start Indices Map: ", window_start_indices[1:5])

--- SLIDING WINDOW PIPELINE DIAGNOSTICS ---
Total Source Timesteps     : 1469
Generated Windows Count    : 284
Relative Time Matrix Shape : (1, 50, 1)
Tensor Windows Array Shape : (4, 50, 284)
Saved Tracking Anchors     : 284

First 5 Window Absolute Start Indices Map: [1, 6, 11, 16, 21]
